# Campaign 4 Fed-Batch Feeding Strategy Comparison

This notebook loads the latest Campaign 4 fed-batch pickle files, extracts projected KPIs, and compares feeding strategies using continuous 960-hour trajectories, metabolite overlays, KPI heatmaps, and a productivity-versus-medium-efficiency trade-off plot.

In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd().resolve()
if ROOT.name == "Cross_Campaign_Comparison":
    ROOT = ROOT.parent
CAMPAIGN_DIR = ROOT / "Campaign_4_mode_comparison_excesskinetics"
FEDBATCH_DIR = CAMPAIGN_DIR / "01_FedBatch"
EXPORT_DIR = FEDBATCH_DIR / "export"
RESULT_SWEEP = "p2"  # Set to "s1" to compare the first optimization sweep instead.
TARGET_HORIZON_H = 960.0
INCLUDE_SETUP_TIME = True
BASELINE_STRATEGY = "Constant"
STRATEGY_ORDER = ["Constant", "Linear", "Exponential", "CSFR", "Glucose Setpoint"]
STRATEGY_COLORS = {"Constant": "#1f77b4", "Linear": "#ff7f0e", "Exponential": "#2ca02c", "CSFR": "#9467bd", "Glucose Setpoint": "#d62728"}
SIMULATION_TO_STRATEGY = {
    "fedbatch_constant": "Constant",
    "fedbatch_linear": "Linear",
    "fedbatch_exponential": "Exponential",
    "fedbatch_csfr": "CSFR",
    "fedbatch_gsp": "Glucose Setpoint",
}


def export_metadata(path):
    """Read the simulation ID and sweep from the timestamped export folder.

    Current Campaign 4 exports are stored as:
    ``export/<simulation_id>_<sweep>_<YYYYMMDD>_<HHMM>/opt_<YYYYMMDD>_<HHMM>.pkl``.
    """
    try:
        simulation_id, sweep, date, time = path.parent.name.rsplit("_", 3)
    except ValueError:
        return None
    if simulation_id not in SIMULATION_TO_STRATEGY or not date.isdigit() or not time.isdigit():
        return None
    return {"simulation_id": simulation_id, "sweep": sweep, "timestamp": f"{date}_{time}"}


def latest_pickles(export_dir, sweep):
    """Return the newest pickle for each requested fed-batch simulation and sweep."""
    latest = {}
    for path in export_dir.rglob("*.pkl"):
        metadata = export_metadata(path)
        if metadata is None or metadata["sweep"] != sweep:
            continue
        simulation_id = metadata["simulation_id"]
        if simulation_id not in latest or path.stat().st_mtime > latest[simulation_id]["path"].stat().st_mtime:
            latest[simulation_id] = {**metadata, "path": path}
    return latest


def simulation_data(obj):
    return obj.data if hasattr(obj, "data") else obj


latest = latest_pickles(EXPORT_DIR, RESULT_SWEEP)
loaded = {}
load_errors = {}
for simulation_id, item in latest.items():
    strategy = SIMULATION_TO_STRATEGY[simulation_id]
    try:
        with item["path"].open("rb") as handle:
            loaded[strategy] = {
                **item,
                "strategy": strategy,
                "data": simulation_data(pickle.load(handle)),
            }
    except Exception as exc:
        load_errors[strategy] = f"{type(exc).__name__}: {exc}"

missing_simulations = [
    simulation_id
    for simulation_id, strategy in SIMULATION_TO_STRATEGY.items()
    if strategy not in loaded and strategy not in load_errors
]

KPI_KEYS = ["STY_kg_m3_day", "total_DCM_harvested_kg", "medium_conversion_efficiency_m3_kg", "feed_conversion_efficiency_kg_kg", "protein_conversion_efficiency_kg_kg", "total_simulation_time_h", "total_base_in_L", "total_acid_in_L", "total_medium_in_L", "number_of_batches", "projection_window_h", "projection_target_time_h", "projection_scale_factor", "last_batch_termination_reason"]


def extract_kpis(data):
    projected = getattr(data, "projected_totals_stats", {}) or {}
    summary = getattr(data, "summary_stats", {}) or {}
    row = {key: projected.get(key, summary.get(key, np.nan)) for key in KPI_KEYS}
    row["projection_mode"] = projected.get("projection_mode", summary.get("projection_mode"))
    return row


kpi_rows = []
for strategy, item in loaded.items():
    row = extract_kpis(item["data"])
    row.update(
        strategy=strategy,
        simulation_id=item["simulation_id"],
        sweep=item["sweep"],
        pickle=str(item["path"]),
    )
    kpi_rows.append(row)
kpi_df = pd.DataFrame(kpi_rows)
if not kpi_df.empty:
    kpi_df["strategy"] = pd.Categorical(kpi_df["strategy"], STRATEGY_ORDER, ordered=True)
    kpi_df = kpi_df.sort_values("strategy").reset_index(drop=True)


def concatenate_profiles(data, include_setup_time=True, max_time_h=None):
    times, states, batch_ids, boundaries = [], [], [], []
    offset = 0.0
    for batch_no, batch in enumerate(data.batches, 1):
        t = np.asarray(batch.t_points, dtype=float) + offset
        y = np.asarray(batch.y_points, dtype=float)
        if max_time_h is not None:
            keep = t <= max_time_h
            t, y = t[keep], y[:, keep]
        if not len(t):
            continue
        times.append(t)
        states.append(y)
        batch_ids.append(np.full(len(t), batch_no))
        boundaries.append(float(t[-1]))
        offset = float(t[-1]) + (float(getattr(batch, "setup_time_after_h", 0) or 0) if include_setup_time else 0)
        if max_time_h is not None and offset >= max_time_h:
            break
    if not times:
        raise ValueError("No batch profile data available")
    return {"time_h": np.concatenate(times), "states": np.concatenate(states, axis=1), "batch_id": np.concatenate(batch_ids), "batch_boundaries_h": boundaries}


profiles = {}
profile_errors = {}
for strategy, item in loaded.items():
    try:
        profiles[strategy] = concatenate_profiles(item["data"], INCLUDE_SETUP_TIME, TARGET_HORIZON_H)
    except Exception as exc:
        profile_errors[strategy] = f"{type(exc).__name__}: {exc}"
STATE = {"Xv": 0, "C_Glc": 1, "C_Gln": 2, "C_Lac": 3, "C_Amm": 4, "V": 5}


def add_boundaries(fig, profile):
    for boundary in profile["batch_boundaries_h"][:-1]:
        if boundary < TARGET_HORIZON_H:
            fig.add_vline(x=boundary, line_width=1, line_dash="dot", line_color="rgba(0,0,0,0.18)")


print(f"Campaign directory: {CAMPAIGN_DIR}")
print(f"Fed-batch export directory: {EXPORT_DIR}")
print(f"Selected optimization sweep: {RESULT_SWEEP}")
print(f"Loaded strategies: {[strategy for strategy in STRATEGY_ORDER if strategy in loaded]}")
if missing_simulations:
    print("Missing simulations:", ", ".join(missing_simulations))
if load_errors:
    print("Load errors:", load_errors)
if profile_errors:
    print("Profile errors:", profile_errors)
if not profiles:
    raise ValueError(f"No fed-batch profiles were loaded for sweep '{RESULT_SWEEP}'.")

fig_density = go.Figure()
for strategy in STRATEGY_ORDER:
    if strategy not in profiles:
        continue
    p = profiles[strategy]
    fig_density.add_trace(go.Scatter(x=p["time_h"], y=p["states"][STATE["Xv"]] / 1e9, mode="lines", name=strategy, line=dict(color=STRATEGY_COLORS[strategy], width=2.8)))
add_boundaries(fig_density, next(iter(profiles.values())))
fig_density.update_layout(title="Campaign 4 Fed-Batch Strategies: Viable Cell Density", xaxis_title="Campaign time (h)", yaxis_title="Viable cell density (10⁶ cells/mL)", xaxis_range=[0, TARGET_HORIZON_H], hovermode="x unified", template="plotly_white", width=1150, height=600)
fig_density.show()

if BASELINE_STRATEGY in profiles:
    base = profiles[BASELINE_STRATEGY]
    fig_delta = go.Figure()
    base_xv = base["states"][STATE["Xv"]] / 1e9
    for strategy in STRATEGY_ORDER:
        if strategy == BASELINE_STRATEGY or strategy not in profiles:
            continue
        p = profiles[strategy]
        xv = np.interp(base["time_h"], p["time_h"], p["states"][STATE["Xv"]] / 1e9)
        fig_delta.add_trace(go.Scatter(x=base["time_h"], y=xv - base_xv, mode="lines", name=strategy, line=dict(color=STRATEGY_COLORS[strategy], width=2.2)))
    fig_delta.add_hline(y=0, line_width=1.2, line_color="rgba(0,0,0,0.5)")
    fig_delta.update_layout(title="Viable Cell Density Difference Relative to Constant Feeding", xaxis_title="Campaign time (h)", yaxis_title="Difference versus Constant (10⁶ cells/mL)", xaxis_range=[0, TARGET_HORIZON_H], hovermode="x unified", template="plotly_white", width=1150, height=550)
    fig_delta.show()

fig_process = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07, subplot_titles=("Viable cell density", "Glucose and lactate", "Glutamine and ammonia"))
for strategy in STRATEGY_ORDER:
    if strategy not in profiles:
        continue
    p = profiles[strategy]
    t = p["time_h"]
    y = p["states"]
    color = STRATEGY_COLORS[strategy]
    for row, idx, name, dash, showlegend in [(1, STATE["Xv"], strategy, "solid", True), (2, STATE["C_Glc"], f"{strategy} — Glucose", "solid", False), (2, STATE["C_Lac"], f"{strategy} — Lactate", "dash", False), (3, STATE["C_Gln"], f"{strategy} — Glutamine", "solid", False), (3, STATE["C_Amm"], f"{strategy} — Ammonia", "dash", False)]:
        values = y[idx] / 1e9 if row == 1 else y[idx]
        fig_process.add_trace(go.Scatter(x=t, y=values, mode="lines", name=name, showlegend=showlegend, legendgroup=strategy, line=dict(color=color, width=2.5 if row == 1 else 1.7, dash=dash)), row=row, col=1)
fig_process.update_yaxes(title_text="10⁶ cells/mL", row=1, col=1)
fig_process.update_yaxes(title_text="mmol/L", row=2, col=1)
fig_process.update_yaxes(title_text="mmol/L", row=3, col=1)
fig_process.update_xaxes(title_text="Campaign time (h)", range=[0, TARGET_HORIZON_H], row=3, col=1)
fig_process.update_layout(title="Campaign 4 Fed-Batch Strategy Trajectories", hovermode="x unified", template="plotly_white", width=1150, height=900)
fig_process.show()

HEATMAP_KPIS = ["STY_kg_m3_day", "total_DCM_harvested_kg", "medium_conversion_efficiency_m3_kg", "feed_conversion_efficiency_kg_kg", "protein_conversion_efficiency_kg_kg"]
KPI_LABELS = {"STY_kg_m3_day": "STY", "total_DCM_harvested_kg": "Harvested DCM", "medium_conversion_efficiency_m3_kg": "Medium efficiency", "feed_conversion_efficiency_kg_kg": "Feed efficiency", "protein_conversion_efficiency_kg_kg": "Protein efficiency"}
if not kpi_df.empty:
    heat = kpi_df.set_index("strategy")[HEATMAP_KPIS].apply(pd.to_numeric, errors="coerce").reindex(STRATEGY_ORDER).dropna(how="all")
    if not heat.empty:
        heat_norm = heat.divide(heat.max().replace(0, 1), axis=1)
        fig_heatmap = go.Figure(go.Heatmap(z=heat_norm.values, x=[KPI_LABELS[k] for k in heat_norm.columns], y=heat_norm.index.astype(str), text=heat.round(3).values, texttemplate="%{text}", colorscale="YlGn", zmin=0, zmax=1, colorbar=dict(title="Relative score")))
        fig_heatmap.update_layout(title="Campaign 4 Fed-Batch Strategy Performance Profile", template="plotly_white", width=1000, height=500)
        fig_heatmap.show()

tradeoff = kpi_df.copy()
for col in ["STY_kg_m3_day", "medium_use_efficiency_kg_m3"]:
    if col in tradeoff:
        tradeoff[col] = pd.to_numeric(tradeoff[col], errors="coerce")
if not tradeoff.empty and {"STY_kg_m3_day", "medium_use_efficiency_kg_m3"}.issubset(tradeoff.columns):
    tradeoff = tradeoff.dropna(subset=["STY_kg_m3_day", "medium_use_efficiency_kg_m3"])
    fig_tradeoff = go.Figure()
    for _, row in tradeoff.iterrows():
        strategy = str(row["strategy"])
        fig_tradeoff.add_trace(go.Scatter(x=[row["medium_use_efficiency_kg_m3"]], y=[row["STY_kg_m3_day"]], mode="markers+text", text=[strategy], textposition="top center", name=strategy, showlegend=False, marker=dict(size=14, color=STRATEGY_COLORS[strategy], line=dict(color="white", width=1))))
    fig_tradeoff.update_layout(title="Fed-Batch Strategy Trade-off: Productivity versus Medium Efficiency", xaxis_title="Medium-use efficiency (kg/m³)", yaxis_title="STY (kg/m³/day)", template="plotly_white", width=900, height=600)
    fig_tradeoff.show()

display(kpi_df)

Campaign directory: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics
Fed-batch export directory: C:\Users\btumulero\OneDrive - Delft University of Technology\Documents\01. Code\Campaign_4_mode_comparison_excesskinetics\01_FedBatch\export
Selected optimization sweep: p2
Loaded strategies: ['Constant', 'Linear', 'Exponential', 'CSFR', 'Glucose Setpoint']


,STY_kg_m3_day,total_DCM_harvested_kg,medium_conversion_efficiency_m3_kg,feed_conversion_efficiency_kg_kg,protein_conversion_efficiency_kg_kg,total_simulation_time_h,total_base_in_L,total_acid_in_L,total_medium_in_L,number_of_batches,projection_window_h,projection_target_time_h,projection_scale_factor,last_batch_termination_reason,projection_mode,strategy,simulation_id,sweep,pickle
0,0.108738,69.592374,2.092924,23.154317,28.678525,7680.0,0.000000,0.0,145651.546391,88.0,960.0,7680.0,8.0,Maximum time reached,yearly_projection,Constant,fedbatch_constant,p2,C:\Users\btumulero\OneDrive - Delft University...
1,0.102855,65.827486,2.647338,18.830131,22.046352,7680.0,0.000000,0.0,174267.584678,112.0,960.0,7680.0,8.0,Maximum time reached,yearly_projection,Linear,fedbatch_linear,p2,C:\Users\btumulero\OneDrive - Delft University...
2,0.113121,72.397433,2.312539,49.150556,68.644341,7680.0,0.000000,0.0,167421.898236,104.0,960.0,7680.0,8.0,Maximum time reached,yearly_projection,Exponential,fedbatch_exponential,p2,C:\Users\btumulero\OneDrive - Delft University...
3,0.094249,60.319149,2.354293,36.509024,49.821037,7680.0,0.000000,0.0,142008.946736,88.0,960.0,7680.0,8.0,Maximum time reached,yearly_projection,CSFR,fedbatch_csfr,p2,C:\Users\btumulero\OneDrive - Delft University...
4,0.005473,3.502493,4.459085,239.040819,264.093042,7680.0,236.508461,0.0,15617.912697,8.0,960.0,7680.0,8.0,Maximum time reached,yearly_projection,Glucose Setpoint,fedbatch_gsp,p2,C:\Users\btumulero\OneDrive - Delft University...


## Single-batch comparison

The campaign-level plots above concatenate all batches across 960 h. The following figures isolate one batch from each strategy so that the feeding-strategy effects can be compared without campaign-scale batch transitions. Change `BATCH_NUMBER` to inspect another corresponding batch.

In [2]:
BATCH_NUMBER = 1


def extract_single_batch_profile(data, batch_number):
    if batch_number < 1 or batch_number > len(data.batches):
        raise IndexError(f"Batch {batch_number} is unavailable; data contains {len(data.batches)} batches.")
    batch = data.batches[batch_number - 1]
    return {
        "batch_number": batch_number,
        "time_h": np.asarray(batch.t_points, dtype=float),
        "states": np.asarray(batch.y_points, dtype=float),
        "termination_reason": getattr(batch, "termination_reason", "Unknown"),
        "t_end_h": float(getattr(batch, "t_end", batch.t_points[-1])),
    }

single_batch_profiles = {}
single_batch_errors = {}
for strategy, item in loaded.items():
    try:
        single_batch_profiles[strategy] = extract_single_batch_profile(item["data"], BATCH_NUMBER)
    except Exception as exc:
        single_batch_errors[strategy] = f"{type(exc).__name__}: {exc}"

print(f"Comparing batch {BATCH_NUMBER} across strategies")
for strategy in STRATEGY_ORDER:
    if strategy in single_batch_profiles:
        profile = single_batch_profiles[strategy]
        print(f"  ✓ {strategy:<18} duration = {profile['t_end_h']:.2f} h")
    elif strategy in single_batch_errors:
        print(f"  - {strategy:<18} {single_batch_errors[strategy]}")

Comparing batch 1 across strategies
  ✓ Constant           duration = 128.11 h
  ✓ Linear             duration = 94.43 h
  ✓ Exponential        duration = 103.93 h
  ✓ CSFR               duration = 120.51 h
  ✓ Glucose Setpoint   duration = 960.00 h


In [3]:
# Single-batch viable cell-density overlay
fig_single_density = go.Figure()
for strategy in STRATEGY_ORDER:
    if strategy not in single_batch_profiles:
        continue
    profile = single_batch_profiles[strategy]
    fig_single_density.add_trace(
        go.Scatter(
            x=profile["time_h"],
            y=profile["states"][STATE["Xv"]] / 1e9,
            mode="lines",
            name=strategy,
            line=dict(color=STRATEGY_COLORS[strategy], width=2.8),
            hovertemplate=(
                f"<b>{strategy}</b><br>"
                "Time: %{x:.1f} h<br>"
                "Viable cell density: %{y:.3f} 10⁶ cells/mL"
                "<extra></extra>"
            ),
        )
    )

fig_single_density.update_layout(
    title=f"Campaign 4 Fed-Batch Strategies — Batch {BATCH_NUMBER}",
    xaxis_title="Batch time (h)",
    yaxis_title="Viable cell density (10⁶ cells/mL)",
    hovermode="x unified",
    template="plotly_white",
    width=1100,
    height=600,
)
fig_single_density.show()

In [4]:
# Single-batch metabolite overlay
fig_single_metabolites = make_subplots(
    rows=1,
    cols=2,
    shared_xaxes=True,
    subplot_titles=("Glucose and glutamine", "Lactate and ammonia"),
    horizontal_spacing=0.12,
)

for strategy in STRATEGY_ORDER:
    if strategy not in single_batch_profiles:
        continue
    profile = single_batch_profiles[strategy]
    t = profile["time_h"]
    y = profile["states"]
    color = STRATEGY_COLORS[strategy]

    for col, idx, name, dash, showlegend in [
        (1, STATE["C_Glc"], strategy + " — Glucose", "solid", True),
        (1, STATE["C_Gln"], strategy + " — Glutamine", "dash", False),
        (2, STATE["C_Lac"], strategy + " — Lactate", "solid", False),
        (2, STATE["C_Amm"], strategy + " — Ammonia", "dash", False),
    ]:
        fig_single_metabolites.add_trace(
            go.Scatter(
                x=t,
                y=y[idx],
                mode="lines",
                name=name,
                legendgroup=strategy,
                showlegend=showlegend,
                line=dict(color=color, width=2, dash=dash),
            ),
            row=1,
            col=col,
        )

fig_single_metabolites.update_xaxes(title_text="Batch time (h)", row=1, col=1)
fig_single_metabolites.update_xaxes(title_text="Batch time (h)", row=1, col=2)
fig_single_metabolites.update_yaxes(title_text="Glc/Gln (mmol/L)", row=1, col=1)
fig_single_metabolites.update_yaxes(title_text="Lactate/Ammonia (mmol/L)", row=1, col=2)
fig_single_metabolites.update_layout(
    title=f"Campaign 4 Fed-Batch Metabolites — Batch {BATCH_NUMBER}",
    hovermode="x unified",
    template="plotly_white",
    width=1200,
    height=550,
)
fig_single_metabolites.show()

In [5]:
# Single-batch process comparison: biomass, metabolites, and volume
fig_single_process = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Viable cell density", "Glucose and lactate", "Culture volume"),
)

for strategy in STRATEGY_ORDER:
    if strategy not in single_batch_profiles:
        continue
    profile = single_batch_profiles[strategy]
    t = profile["time_h"]
    y = profile["states"]
    color = STRATEGY_COLORS[strategy]

    for row, idx, name, dash, showlegend in [
        (1, STATE["Xv"], strategy, "solid", True),
        (2, STATE["C_Glc"], strategy + " — Glucose", "solid", False),
        (2, STATE["C_Lac"], strategy + " — Lactate", "dash", False),
        (3, STATE["V"], strategy + " — Volume", "solid", False),
    ]:
        values = y[idx] / 1e9 if row == 1 else y[idx]
        fig_single_process.add_trace(
            go.Scatter(
                x=t,
                y=values,
                mode="lines",
                name=name,
                legendgroup=strategy,
                showlegend=showlegend,
                line=dict(color=color, width=2.6 if row == 1 else 1.8, dash=dash),
            ),
            row=row,
            col=1,
        )

fig_single_process.update_yaxes(title_text="10⁶ cells/mL", row=1, col=1)
fig_single_process.update_yaxes(title_text="mmol/L", row=2, col=1)
fig_single_process.update_yaxes(title_text="Volume (L)", row=3, col=1)
fig_single_process.update_xaxes(title_text="Batch time (h)", row=3, col=1)
fig_single_process.update_layout(
    title=f"Campaign 4 Fed-Batch Process Profiles — Batch {BATCH_NUMBER}",
    hovermode="x unified",
    template="plotly_white",
    width=1100,
    height=850,
)
fig_single_process.show()